In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
enron = pd.read_csv('data/enron_email_context/EnronEmailReplyPairsWithContext.csv')

In [4]:
enron.head(10)

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context
0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,6/18/2001 22:44,6/19/2001 15:49,NaN
1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,3/19/2002 8:30,3/19/2002 8:34,NaN
2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,1/15/2002 12:30,1/15/2002 16:29,NaN
3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,1/19/2002 10:55,1/19/2002 10:56,NaN
4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,1/21/2002 15:30,1/21/2002 15:44,NaN
5,What are they having?,Need you or John to step out. : Louise.Kitchen...,RE:,Re: Did you sneak out?,louise.kitchen@enron.com,8777865122@skytel.com,1/21/2002 16:26,1/23/2002 9:24,NaN
6,What is the situation with a lawsuit involving...,Solarc is the system used for NGLs etc - we do...,Solarc,RE: Solarc,8777865122@skytel.com,louise.kitchen@enron.com,9/1/2002 8:23,9/1/2002 8:27,NaN
7,Did you get tg back to 1yr docs,Did not get a commitment. Just told them I tho...,Re: Solarc,Re: Solarc,louise.kitchen@enron.com,8777865122@skytel.com,10/1/2002 10:32,10/1/2002 11:00,NaN
8,What do you think happens if they require 2yrs?,50% or less of the current group. We have a ma...,Contracts,RE: Contracts,8777865122@skytel.com,louise.kitchen@enron.com,10/1/2002 11:10,10/1/2002 11:15,NaN
9,We are going to have to work on making this mo...,"I understand. If I have too, I will dip into m...",Money,RE: Money,8777865122@skytel.com,m..presto@enron.com,1/19/2002 10:31,1/19/2002 15:49,NaN


In [7]:
# Use Regex to parse out any unwanted characters
enron['DateSend'] = enron['DateSend'].astype(str).str.replace(r'[^\w\s,:/:-]', '', regex=True).str.strip()
enron['DateReply'] = enron['DateReply'].astype(str).str.replace(r'[^\w\s,:/:-]', '', regex=True).str.strip()

# Convert date columns to datetime objects and fill w NaN values
enron['DateSend'] = pd.to_datetime(enron['DateSend'], format='mixed', errors='coerce')
enron['DateReply'] = pd.to_datetime(enron['DateReply'], format='mixed', errors='coerce')

# Calculate if reply was within 30 minutes and convert True/False to 1/0
enron['Response_Within_30_Mins'] = (
    (enron['DateReply'] - enron['DateSend']) <= pd.Timedelta(minutes=30)
).astype(int)

# Check the breakdown of 1s and 0s
enron['Response_Within_30_Mins'].value_counts()

Response_Within_30_Mins
0    8416
1    6961
Name: count, dtype: int64

In [9]:
# Extract the hour of the day (0-23) - people don't reply fast at 3 AM!
enron['HourSent'] = enron['DateSend'].dt.hour

# Extract day of the week (0=Monday, 6=Sunday)
enron['DaySent'] = enron['DateSend'].dt.dayofweek

# Create a binary flag for weekends
enron['IsWeekend'] = enron['DaySent'].isin([5, 6]).astype(int)

In [10]:
enron.head()

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,Response_Within_30_Mins,HourSent,DaySent,IsWeekend
0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,2001-06-18 22:44:00,2001-06-19 15:49:00,NaN,0,22.0,0.0,0
1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,2002-03-19 08:30:00,2002-03-19 08:34:00,NaN,1,8.0,1.0,0
2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,2002-01-15 12:30:00,2002-01-15 16:29:00,NaN,0,12.0,1.0,0
3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,2002-01-19 10:55:00,2002-01-19 10:56:00,NaN,1,10.0,5.0,1
4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,2002-01-21 15:30:00,2002-01-21 15:44:00,NaN,1,15.0,0.0,0
